
# Titanic Survival Prediction – Decision Tree  
## Pandas 3.x + Cross‑Validation + GridSearchCV

This notebook is **fully compatible with pandas 3.x**:
- No deprecated `inplace=True`
- No chained assignment
- Explicit copies where required

Topics covered:
- Titanic dataset loading
- Preprocessing with pandas 3.x
- Decision Tree model
- **k‑Fold Cross‑Validation**
- **Hyperparameter Optimization using GridSearchCV**
- Evaluation metrics
- Tree visualization



## How Cross‑Validation Works (Intuition)

**k‑Fold Cross‑Validation**
1. Split data into `k` equal folds
2. Train on `k‑1` folds
3. Validate on remaining fold
4. Repeat `k` times
5. Final score = average of all folds

**Why it matters**
- Reduces overfitting
- More stable than single train/test split
- Used internally by GridSearchCV


In [ ]:

# Library imports (pandas 3.x compatible)
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt


In [ ]:

# Load Titanic dataset from OpenML
titanic = fetch_openml(name="titanic", version=1, as_frame=True)
df = titanic.frame.copy()

df.head()


In [ ]:

# Select required columns
features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
target = "survived"

df = df[features + [target]].copy()

# Type casting
df["age"] = df["age"].astype("float64")
df["fare"] = df["fare"].astype("float64")
df[target] = df[target].astype("int64")

# Handle missing values (pandas 3 safe)
df["age"] = df["age"].fillna(df["age"].median())
df["fare"] = df["fare"].fillna(df["fare"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])

df.isna().sum()


In [ ]:

# One‑hot encoding categorical features
df = pd.get_dummies(
    df,
    columns=["sex", "embarked"],
    drop_first=True
)

df.head()


In [ ]:

# Feature matrix and target vector
X = df.drop(columns=[target])
y = df[target]

# Train‑test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



## Baseline Decision Tree


In [ ]:

# Baseline model
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))



## GridSearchCV – Hyperparameter Optimization

GridSearchCV:
- Tries every parameter combination
- Uses **k‑fold cross‑validation**
- Selects model with best average CV score


In [ ]:

param_grid = {
    "max_depth": [None, 3, 5, 7, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "criterion": ["gini", "entropy"]
}

grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid.fit(X_train, y_train)

grid.best_params_, grid.best_score_


In [ ]:

# Best model evaluation
best_dt = grid.best_estimator_

y_pred_best = best_dt.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_best))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best))



## Decision Tree Visualization


In [ ]:

plt.figure(figsize=(20, 10))
plot_tree(
    best_dt,
    feature_names=X.columns,
    class_names=["Not Survived", "Survived"],
    filled=True,
    max_depth=3
)
plt.show()



## Summary
- Pandas 3.x safe preprocessing
- Cross‑validation reduces variance
- GridSearchCV finds optimal tree depth & splits
- Constrained trees generalize better than deep trees
